# Lab 12 · Reference solution

The polished final implementation of [Lab 12: Plan-and-execute from scratch](../README.md).

A planner emits a structured `Plan` (typed `PlanStep` list with `depends_on` and `parallel_group`); a supervisor dispatches steps to a bounded executor pool (`ThreadPoolExecutor(max_workers=3)`); failures trigger bounded replanning (`MAX_REPLANS = 2`) with plan-signature dedup.

This notebook is the reference implementation; refer to [`../lab.ipynb`](../lab.ipynb) for the pedagogical step-by-step build. The [`solution README`](./README.md) covers implementation choices, common variations, and bugs to watch for.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Lab 10 machinery (chat client + web tools + helpers)

Unchanged. Repeated here for self-containment.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(messages: list[dict], tools: list[dict] | None = None,
                     tool_choice: str = "auto", temperature: float = 0) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"], "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=2048, temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
               for b in resp.content if getattr(b, "type", None) == "tool_use"]
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    return raw


# Web tools (unchanged from Lab 10)
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit", "register to read",
]


def web_search(query: str, recency: RecencyType = "any", max_results: int = 8) -> dict:
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(query=query.strip(), region="us-en", safesearch="moderate",
                            timelimit=_RECENCY_MAP.get(recency),
                            max_results=max_results, backend="auto")
    except (RatelimitException, TimeoutException, DDGSException) as e:
        kind = {"RatelimitException": "rate_limit", "TimeoutException": "timeout"}.get(
            type(e).__name__, "other")
        return {"status": "error", "kind": kind, "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": f"{type(e).__name__}: {e}"}
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results"}
    return {"status": "ok",
            "results": [{"title": (r.get("title") or "").strip(),
                         "url": (r.get("href") or "").strip(),
                         "snippet": (r.get("body") or "").strip()}
                        for r in raw if r.get("href")][:max_results]}


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout", "detail": "timeout"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected"}
    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


## `PlanStep` and `Plan` schemas

Strict Pydantic models with full graph validation. `validate_graph()` catches cycles (Kahn's algorithm), duplicate IDs, unknown tools, unknown dependency references, and parallel-group integrity violations in one pass.

In [ ]:
class PlanStep(StrictModel):
    id: str = Field(description="Unique step ID, e.g. 'step_1'")
    description: str = Field(description="What this step does, self-contained.")
    tool: str = Field(description="Tool to invoke. Must be in executor registry.")
    args: dict = Field(default_factory=dict, description="Arguments to the tool.")
    depends_on: list[str] = Field(default_factory=list,
                                    description="IDs of steps whose output this step needs.")
    parallel_group: str | None = Field(
        default=None,
        description="Optional group for concurrent execution. None = run alone.",
    )


class Plan(StrictModel):
    steps: list[PlanStep] = Field(description="The ordered list of steps.")

    def validate_graph(self, available_tools: set[str]) -> list[str]:
        """Return a list of validation errors (empty list = valid plan)."""
        errors: list[str] = []
        step_ids = {s.id for s in self.steps}
        if len(step_ids) != len(self.steps):
            errors.append("duplicate step IDs")
        for s in self.steps:
            if s.tool not in available_tools:
                errors.append(
                    f"step {s.id} uses tool '{s.tool}' not in executor registry: "
                    f"{sorted(available_tools)}"
                )
            for dep in s.depends_on:
                if dep not in step_ids:
                    errors.append(f"step {s.id} depends on unknown step '{dep}'")
                if dep == s.id:
                    errors.append(f"step {s.id} depends on itself")
        # Parallel groups: no internal dependencies
        by_group: dict[str, list[PlanStep]] = {}
        for s in self.steps:
            if s.parallel_group is not None:
                by_group.setdefault(s.parallel_group, []).append(s)
        for group_name, group_steps in by_group.items():
            group_ids = {s.id for s in group_steps}
            for s in group_steps:
                for dep in s.depends_on:
                    if dep in group_ids:
                        errors.append(
                            f"parallel_group '{group_name}' contains step {s.id} "
                            f"which depends on group-mate {dep}"
                        )
        # Cycle detection (Kahn's algorithm)
        incoming = {s.id: set(s.depends_on) for s in self.steps}
        no_incoming = [sid for sid, deps in incoming.items() if not deps]
        visited: list[str] = []
        while no_incoming:
            n = no_incoming.pop()
            visited.append(n)
            for sid, deps in incoming.items():
                if n in deps:
                    deps.discard(n)
                    if not deps and sid not in visited and sid not in no_incoming:
                        no_incoming.append(sid)
        if len(visited) < len(self.steps):
            unvisited = [s.id for s in self.steps if s.id not in visited]
            errors.append(f"cycle detected involving steps: {unvisited}")
        return errors


MAX_PLAN_STEPS = 8
MAX_PARALLEL_EXECUTORS = 3
EXECUTOR_MAX_STEPS = 4
MAX_REPLANS = 2
SUPERVISOR_MAX_STEPS = 12

print(f"Caps: MAX_PLAN_STEPS={MAX_PLAN_STEPS}, "
      f"MAX_PARALLEL_EXECUTORS={MAX_PARALLEL_EXECUTORS}, "
      f"EXECUTOR_MAX_STEPS={EXECUTOR_MAX_STEPS}, "
      f"MAX_REPLANS={MAX_REPLANS}, "
      f"SUPERVISOR_MAX_STEPS={SUPERVISOR_MAX_STEPS}")


## Planner agent

Emits JSON-validated `Plan`. Five planner-prompt rules in the system prompt (atomic steps, explicit dependencies, practical parallel groups, self-contained descriptions, bounded plans). The executor's tool registry is passed into the system prompt — this closes the plan-execution gap structurally. Retry loop surfaces validation errors back to the planner.

In [ ]:
EXECUTOR_TOOL_REGISTRY: dict[str, dict] = {
    "web_search": {
        "description": (
            "Search the web. Returns up to max_results items with title, url, "
            "snippet."
        ),
        "args_schema": {
            "query": "string, 3-8 specific words",
            "recency": "one of: any, day, week, month, year (default: any)",
            "max_results": "integer 1-10 (default: 8)",
        },
    },
    "fetch_page": {
        "description": "Fetch the full text content of a single URL.",
        "args_schema": {
            "url": "string, the URL to fetch",
            "max_chars": "integer (default: 8000)",
        },
    },
}


def _format_tool_registry(registry: dict[str, dict]) -> str:
    lines = []
    for tool_name, info in registry.items():
        lines.append(f"- {tool_name}: {info['description']}")
        for arg_name, arg_desc in info["args_schema"].items():
            lines.append(f"    args.{arg_name}: {arg_desc}")
    return "\n".join(lines)


PLANNER_SYSTEM_PROMPT_TEMPLATE = """You are a planner agent. Given a user task,
emit a Plan as a JSON object:

{{
  "steps": [
    {{
      "id": "step_1",
      "description": "What this step does, self-contained.",
      "tool": "tool_name",
      "args": {{...}},
      "depends_on": ["step_id_1", ...],
      "parallel_group": "group_name" or null
    }},
    ...
  ]
}}

EXECUTOR has access to these tools (and only these):
{tool_registry}

RULES:

1. ATOMIC STEPS. One tool call per step.
2. EXPLICIT DEPENDENCIES. List all step IDs whose output you use in depends_on.
3. HONEST PARALLEL GROUPS. Steps sharing a parallel_group must NOT depend on
   each other (directly or transitively).
4. SELF-CONTAINED DESCRIPTIONS. Executor sees only one step + its deps' outputs.
5. BOUNDED PLANS. At most {max_steps} steps.

Return ONLY JSON. No prose preamble. No markdown fences.
"""


def planner_agent(task: str, failure_context: dict | None = None,
                   max_retries: int = 2) -> Plan | dict:
    system_prompt = PLANNER_SYSTEM_PROMPT_TEMPLATE.format(
        tool_registry=_format_tool_registry(EXECUTOR_TOOL_REGISTRY),
        max_steps=MAX_PLAN_STEPS,
    )
    user_prompt = f"USER TASK:\n{task}"
    if failure_context:
        user_prompt += (
            f"\n\nA PREVIOUS PLAN FAILED. Revise:\n"
            f"failed_step_id: {failure_context.get('step_id')}\n"
            f"error: {failure_context.get('error')}\n"
            f"completed_steps: {failure_context.get('completed_steps', [])}\n"
            f"Produce a new plan that avoids the failure."
        )

    last_error = "no attempt"
    for _attempt in range(max_retries + 1):
        msg = chat_with_tools(
            [{"role": "system", "content": system_prompt},
             {"role": "user", "content": user_prompt}],
            temperature=0,
        )
        raw = _strip_code_fences(msg.content or "")
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError as e:
            last_error = f"JSON parse error: {e}"
            user_prompt = f"{user_prompt}\n\nPrev response wasn't JSON: {last_error}"
            continue
        try:
            plan = Plan.model_validate(obj)
        except ValidationError as e:
            last_error = f"schema validation error: {e}"
            user_prompt = f"{user_prompt}\n\nSchema validation failed: {last_error}"
            continue
        errors = plan.validate_graph(set(EXECUTOR_TOOL_REGISTRY.keys()))
        if errors:
            last_error = "; ".join(errors)
            user_prompt = f"{user_prompt}\n\nGraph errors: {last_error}"
            continue
        if len(plan.steps) > MAX_PLAN_STEPS:
            last_error = f"plan has {len(plan.steps)} steps; max {MAX_PLAN_STEPS}"
            user_prompt = f"{user_prompt}\n\n{last_error}"
            continue
        return plan

    return {"status": "error", "kind": "planner_failed", "detail": last_error}


## Executor agent

Runs ONE step at a time. Anti-improvement framing in the system prompt. After the LLM emits a tool call, the executor validates `tc.name == step.tool` and returns a `wrong_tool` envelope if the LLM deviates. Supports the `cannot_execute` structured signal.

In [ ]:
EXECUTOR_SYSTEM_PROMPT = """You are an executor worker. You receive ONE step from
a plan plus the outputs of any steps it depends on. RUN the step's specified tool
with its specified arguments.

You MAY:
- Substitute placeholder values in args with concrete values from dependency_outputs
  (e.g. if args.url is "<first URL from step_1>", pick the actual URL).

You MUST NOT:
- Use a different tool than the one specified.
- "Improve" the args beyond filling in placeholders.

If you cannot execute, respond with JSON:
  {"status": "cannot_execute", "reason": "<specific reason>"}

Otherwise invoke the tool ONCE and return its result.
"""


def _make_executor_tool_schemas() -> list[dict]:
    return [
        {"type": "function",
         "function": {"name": "web_search",
                      "description": EXECUTOR_TOOL_REGISTRY["web_search"]["description"],
                      "parameters": {"type": "object",
                                     "properties": {"query": {"type": "string"},
                                                    "recency": {"type": "string",
                                                                "enum": ["any", "day", "week",
                                                                         "month", "year"]},
                                                    "max_results": {"type": "integer"}},
                                     "required": ["query"]}}},
        {"type": "function",
         "function": {"name": "fetch_page",
                      "description": EXECUTOR_TOOL_REGISTRY["fetch_page"]["description"],
                      "parameters": {"type": "object",
                                     "properties": {"url": {"type": "string"},
                                                    "max_chars": {"type": "integer"}},
                                     "required": ["url"]}}},
    ]


def _execute_tool(name: str, args: dict) -> dict:
    if name == "web_search":
        return web_search(args.get("query", ""), args.get("recency", "any"),
                          args.get("max_results", 8))
    if name == "fetch_page":
        return fetch_page(args.get("url", ""), args.get("max_chars", 8000))
    return {"status": "error", "kind": "unknown_tool", "detail": name}


def executor_agent(step: PlanStep, dependency_outputs: dict) -> dict:
    deps_summary = json.dumps(dependency_outputs, indent=2)[:4000]
    user_prompt = (
        f"STEP TO EXECUTE:\n"
        f"  id: {step.id}\n"
        f"  description: {step.description}\n"
        f"  tool: {step.tool}\n"
        f"  args: {json.dumps(step.args)}\n\n"
        f"DEPENDENCY OUTPUTS:\n{deps_summary}\n\n"
        f"Call the specified tool ONCE."
    )
    messages: list[dict] = [
        {"role": "system", "content": EXECUTOR_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    tools = _make_executor_tool_schemas()

    for _ in range(EXECUTOR_MAX_STEPS):
        msg = chat_with_tools(messages, tools=tools)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            raw = _strip_code_fences(msg.content or "")
            try:
                obj = json.loads(raw)
                if isinstance(obj, dict) and obj.get("status") == "cannot_execute":
                    return obj
            except json.JSONDecodeError:
                pass
            return {"status": "error", "kind": "no_tool_call",
                    "detail": "executor returned no tool call"}

        tc = msg.tool_calls[0]
        if tc.name != step.tool:
            return {"status": "error", "kind": "wrong_tool",
                    "detail": f"executor used {tc.name} but step.tool was {step.tool}"}
        result = _execute_tool(tc.name, tc.arguments)
        return {"status": "ok", "tool": tc.name, "args": tc.arguments, "result": result}

    return {"status": "error", "kind": "executor_step_cap",
            "detail": f"exceeded EXECUTOR_MAX_STEPS={EXECUTOR_MAX_STEPS}"}


## Dispatcher (pure Python — not LLM-driven)

Computes ready steps (`depends_on` all completed), submits them to the pool respecting `parallel_group`, collects results. `threading.Lock` around the shared `completed` dict prevents race conditions. Action-hash dedup at the pool level prevents duplicate dispatch.

In [ ]:
def _ready_steps(plan: Plan, completed: dict[str, dict],
                  failed: set[str], in_flight: set[str]) -> list[PlanStep]:
    return [
        s for s in plan.steps
        if (s.id not in completed and s.id not in in_flight and s.id not in failed
            and all(dep in completed for dep in s.depends_on))
    ]


def dispatch_plan(plan: Plan) -> dict:
    completed: dict[str, dict] = {}
    failed_steps: set[str] = set()
    failed_records: list[dict] = []
    in_flight: set[str] = set()
    lock = threading.Lock()
    start = time.time()
    seen_action_hashes: set[str] = set()

    def _run_step(step: PlanStep) -> tuple[str, dict]:
        with lock:
            dep_outputs = {dep: completed[dep] for dep in step.depends_on}
        action_sig = _action_hash(step.tool, step.args)
        if action_sig in seen_action_hashes:
            return step.id, {"status": "error", "kind": "duplicate_action",
                             "detail": f"step {step.id} matches previous action"}
        seen_action_hashes.add(action_sig)
        return step.id, executor_agent(step, dep_outputs)

    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_EXECUTORS) as pool:
        while True:
            ready = _ready_steps(plan, completed, failed_steps, in_flight)
            if not ready and not in_flight:
                break
            futures = []
            for step in ready:
                if len(in_flight) >= MAX_PARALLEL_EXECUTORS:
                    break
                in_flight.add(step.id)
                futures.append(pool.submit(_run_step, step))
            if not futures and in_flight:
                time.sleep(0.05)
                continue
            for fut in as_completed(futures):
                step_id, result = fut.result()
                with lock:
                    in_flight.discard(step_id)
                    if result.get("status") == "ok":
                        completed[step_id] = result
                    else:
                        failed_steps.add(step_id)
                        failed_records.append({"step_id": step_id, **result})

    return {"status": "ok" if not failed_steps else "partial",
            "results": completed, "failed": failed_records,
            "timing": {"wall_clock_s": round(time.time() - start, 2)}}


## Replanning + plan signature

`_plan_signature` hashes structural content only (id, tool, args, sorted deps, parallel_group) — NOT descriptions. Identical-plan replans are escalated to `partial_after_cap`.

In [ ]:
def _plan_signature(plan: Plan) -> str:
    """Stable hash of structural plan content. Excludes descriptions."""
    structural = [
        {"id": s.id, "tool": s.tool, "args": s.args,
         "depends_on": sorted(s.depends_on),
         "parallel_group": s.parallel_group}
        for s in plan.steps
    ]
    return hashlib.sha256(
        json.dumps(structural, sort_keys=True).encode()
    ).hexdigest()[:16]


def plan_and_execute(task: str) -> dict:
    seen_plan_sigs: set[str] = set()
    replan_count = 0
    last_partial: dict | None = None
    failure_context: dict | None = None

    while True:
        plan_or_err = planner_agent(task, failure_context=failure_context)
        if not isinstance(plan_or_err, Plan):
            return {"status": "error", "stage": "planner",
                    "detail": plan_or_err, "partial": last_partial}
        plan = plan_or_err
        sig = _plan_signature(plan)
        if sig in seen_plan_sigs:
            return {"status": "error", "stage": "replanner_duplicate",
                    "detail": "replanner produced identical plan; escalating",
                    "plan": [s.model_dump() for s in plan.steps],
                    "partial": last_partial}
        seen_plan_sigs.add(sig)

        exec_result = dispatch_plan(plan)
        last_partial = exec_result

        if exec_result["status"] == "ok":
            return {"status": "ok",
                    "plan": [s.model_dump() for s in plan.steps],
                    "execution": exec_result,
                    "replans": replan_count}

        if replan_count >= MAX_REPLANS:
            return {"status": "partial_after_cap",
                    "plan": [s.model_dump() for s in plan.steps],
                    "execution": exec_result, "replans": replan_count,
                    "detail": f"Hit MAX_REPLANS={MAX_REPLANS}; surfacing partial."}

        first_failure = exec_result["failed"][0]
        failure_context = {
            "step_id": first_failure["step_id"],
            "error": first_failure.get("detail") or first_failure.get("kind"),
            "completed_steps": list(exec_result["results"].keys()),
        }
        replan_count += 1


## Synthesizer

Composes the final answer from step results. Citation preservation for fetched pages.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer. You receive the original
task plus a dict of executed step results from a plan. Produce a clear answer
using only information from the step results.

Rules:
1. Cite fetched pages inline as [1], [2]; list at end as [N] Title — URL.
2. Do not invent claims not supported by step results.
3. If a step failed, say so. Do not paper over gaps.
"""


def synthesizer_agent(task: str, execution: dict) -> dict:
    results = execution.get("results", {})
    failed = execution.get("failed", [])

    step_summary_lines = []
    citations: list[dict] = []
    for step_id, step_result in results.items():
        tool = step_result.get("tool")
        result = step_result.get("result", {})
        result_str = json.dumps(result)[:3000]
        step_summary_lines.append(f"--- {step_id} ({tool}) ---\n{result_str}")
        if tool == "fetch_page" and result.get("status") in ("ok", "too_long"):
            citations.append({"url": result["url"], "title": result.get("title", "")})

    step_summary = "\n\n".join(step_summary_lines) or "(no completed steps)"
    failure_block = ""
    if failed:
        failure_block = "\n\nFAILED STEPS:\n" + "\n".join(
            f"- {f['step_id']}: {f.get('kind')} ({f.get('detail', '')[:200]})"
            for f in failed
        )

    user_prompt = (
        f"USER TASK:\n{task}\n\n"
        f"COMPLETED STEP RESULTS:\n{step_summary}{failure_block}\n\n"
        f"Compose the final answer. Cite fetched URLs inline."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {"status": "ok", "answer": (msg.content or "").strip(),
            "citations": citations}


def run_plan_and_execute(task: str) -> dict:
    """Top-level entry point: plan → execute → synthesize."""
    result = plan_and_execute(task)
    if result["status"] == "error":
        return {"status": "error", "detail": result.get("detail"),
                "partial": result.get("partial"), "answer": None}
    synth = synthesizer_agent(task, result["execution"])
    return {"status": result["status"], "answer": synth["answer"],
            "citations": synth["citations"], "plan": result["plan"],
            "replans": result.get("replans", 0),
            "wall_clock_s": result["execution"]["timing"]["wall_clock_s"],
            "failed_steps": result["execution"].get("failed", [])}


## Demo

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP). "
    "Summarize the top 3 key points with citations."
)
final = run_plan_and_execute(task)

print(f"Status: {final['status']}, replans: {final['replans']}, "
      f"wall_clock: {final['wall_clock_s']}s, "
      f"plan: {len(final['plan'])} steps, "
      f"failed: {len(final['failed_steps'])}")
print("=" * 70)
print(final["answer"])


**Sample output (LLM responses will vary; trajectory should be stable):**

```
Status: ok, replans: 0, wall_clock: 6.4s, plan: 4 steps, failed: 0
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by
Anthropic in late 2024. Three recent developments stand out:

1. Expanded server ecosystem...
2. Production deployments...
3. Tool authentication patterns...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Specification — https://modelcontextprotocol.io/...
[3] ...
```

A typical clean run: 4 steps (search + 3 parallel fetches), 0 replans, wall-clock dominated by the parallel-fetch group running concurrently rather than serially.

## Production readiness — out of scope here

For a real deployment you'd want: distributed execution (ProcessPoolExecutor or queue+workers); persistent plan state (LangGraph's checkpointer or equivalent); per-step cost budgets and global enforcement; observability at the dispatcher; plan-quality eval (replan rate, plan-step utilization); critic-on-plan or critic-on-synthesis (generator-critic from Lab 11 applied at one of those stages).

The next pattern (Lab 13) composes Path 02's retrieval pipeline with this pattern + Lab 10's supervisor — the integrative module. Same machinery, one new worker role (retriever).